## FFC — Step 4 (Local): Pincode Universe (Unmapped)

This notebook produces a local equivalent of the Step‑4 deliverable generated in `FFC/Step 4 - Pincode Universe.ipynb`.

It uses **local copies** of the Step‑2 final offtake output, Step‑3 Apollo/Keimed DB-level output, and the current pincode→territory universe mapping file.

### Inputs (local)
- Step 2 output CSV (local)
- Step 3 output CSV (local)
- Pincode universe mapping CSV (local)

### Output
- `output/Pincode_Universe_Unmapped_local.csv`


In [52]:
from pathlib import Path

print("Working directory:", Path().resolve())


Working directory: C:\Users\pawarux2\OneDrive - Abbott\Documents\FFC\FFC Test


In [53]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

os.makedirs("output", exist_ok=True)

# ----------------------------
# CONFIG (edit these paths)
# ----------------------------

# Step 2 final offtake file (local)
OFFTAKE_FILE = Path("output/offtake_step2_final.csv")

# Step 3 Apollo+Keimed DB-level file (local)
DB_LEVEL_FILE = Path("output/DB_Level_Apollo_Keimed_2026_local.csv")

# Pincode universe mapping (use the in-repo file by default)
# This file matches the original Step-4 inputs and contains:
# pincode, Division Name, Affiliate, State, District, Previously Mapped Territory Code, Final Mapped Territory
PIN_UNIVERSE_FILE = Path("C:/Users/pawarux2/OneDrive - Abbott/Documents/FFC/FFC Test/input/20260402_Mar Pincode_Universe_AIL Mapped.csv")

OUT_UNMAPPED = Path("output/Pincode_Universe_Unmapped_local.csv")

for p in [OFFTAKE_FILE, DB_LEVEL_FILE, PIN_UNIVERSE_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")


In [54]:
# Load inputs (optimized: read only required columns)

offtake = pd.read_csv(
    OFFTAKE_FILE,
    usecols=["pincode", "Division Name", "Affiliate", "State", "District"],
    dtype={"pincode": "string", "Division Name": "string", "Affiliate": "string"},
    low_memory=False,
)

db_level = pd.read_csv(
    DB_LEVEL_FILE,
    usecols=["pincode", "Division Name", "Affiliate", "State", "District"],
    dtype={"pincode": "string", "Division Name": "string", "Affiliate": "string"},
    low_memory=False,
)

pin_universe = pd.read_csv(
    PIN_UNIVERSE_FILE,
    usecols=[
        "pincode",
        "Division Name",
        "Affiliate",
        "State",
        "District",
        "Previously Mapped Territory Code",
        "Final Mapped Territory",
    ],
    dtype={"pincode": "string", "Division Name": "string", "Affiliate": "string"},
    low_memory=False,
)

offtake.head(2), db_level.head(2), pin_universe.head(2)


(  Affiliate pincode             State   District Division Name
 0       AIL  201003     Uttar Pradesh  Ghaziabad          <NA>
 1       AIL  176115  Himachal Pradesh     Kangra          <NA>,
     pincode Division Name Affiliate        State           District
 0  700104.0     gi optima       AIL  West Bengal  South 24 Parganas
 1  700107.0       insulin      Novo  West Bengal            Kolkata,
   pincode Division Name Affiliate    State   District  \
 0  380058       GENNEXT       AIL  Gujarat  Ahmedabad   
 1  124507       GENNEXT       AIL  Haryana    Jhajjar   
 
   Previously Mapped Territory Code Final Mapped Territory  
 0                         PT001375               PT001375  
 1                         IT010494               IT010494  )

In [55]:
# Clean mapping universe (same logic as original Step-4)
pin_universe_2 = pin_universe.copy()

bad_markers = {"0", "Not mapped", "Unmapped", "Dummy"}

pin_universe_2["Final Mapped Territory"] = np.where(
    pin_universe_2["Final Mapped Territory"].astype(str).isin(bad_markers),
    np.nan,
    pin_universe_2["Final Mapped Territory"],
)

pin_universe_2["Previously Mapped Territory Code"] = np.where(
    pin_universe_2["Previously Mapped Territory Code"].astype(str).isin(bad_markers),
    np.nan,
    pin_universe_2["Previously Mapped Territory Code"],
)

pin_universe_2["Final Mapped Territory"] = np.where(
    pd.isna(pin_universe_2["Final Mapped Territory"]),
    pin_universe_2["Previously Mapped Territory Code"],
    pin_universe_2["Final Mapped Territory"],
)

pin_universe_2.head(10)

,pincode,Division Name,Affiliate,State,District,Previously Mapped Territory Code,Final Mapped Territory
0,380058,GENNEXT,AIL,Gujarat,Ahmedabad,PT001375,PT001375
1,124507,GENNEXT,AIL,Haryana,Jhajjar,IT010494,IT010494
2,277207,GENNEXT,AIL,Uttar Pradesh,Ballia,IT007172,IT007172
3,560043,GENNEXT,AIL,Karnataka,Bangalore,PT001410,PT001410
4,560064,GENNEXT,AIL,Karnataka,Bangalore,PT001410,PT001410
5,560102,GENNEXT,AIL,Karnataka,Bangalore,PT001410,PT001410
6,560103,GENNEXT,AIL,Karnataka,Bangalore,PT001410,PT001410
7,600100,GENNEXT,AIL,Tamil Nadu,Kanchipuram,PT001395,PT001395
8,846005,GENNEXT,AIL,Bihar,Darbhanga,IT007198,IT007198
9,110005,GENNEXT,AIL,Delhi,Delhi,PT001383,PT001383


In [56]:
# Collect pincodes observed in sales (offtake + db-level)

# Observed pincodes from both sources
offtake_obs = offtake.copy()
db_obs = db_level.copy()

observed = pd.concat([offtake_obs, db_obs], ignore_index=True)
observed = observed.drop_duplicates()

observed.shape


(141854, 5)

In [57]:
cols = ["Division Name", "Affiliate"]

for df in [observed, pin_universe_2]:
    for c in cols:
        df[c] = df[c].str.upper().str.strip()

In [58]:
# Left-join to mapping universe to identify missing (unmapped) pincodes
merged = pd.merge(
    observed,
    pin_universe_2[["pincode", "Division Name", "Affiliate", "Final Mapped Territory"]],
    how="left",
    on=["pincode", "Division Name", "Affiliate"],
)

unmapped = merged[pd.isna(merged["Final Mapped Territory"])].copy()

unmapped.to_csv(OUT_UNMAPPED, index=False)

{
    "unmapped_rows": unmapped.shape[0],
    "output": str(OUT_UNMAPPED),
}


{'unmapped_rows': 128367,
 'output': 'output\\Pincode_Universe_Unmapped_local.csv'}

In [59]:
# ---------------------------------------
# MAPPED vs UNMAPPED PINCODE COMPARISON
# ---------------------------------------

# Merge observed pincodes with full mapping universe
comparison = pd.merge(
    observed,
    pin_universe_2[
        ["pincode", "Division Name", "Affiliate", "Final Mapped Territory"]
    ],
    how="left",
    on=["pincode", "Division Name", "Affiliate"],
)

# Create mapping status flag
comparison["Mapping Status"] = np.where(
    comparison["Final Mapped Territory"].notna(),
    "Mapped",
    "Unmapped",
)

# Optional: sort for readability
comparison = comparison.sort_values(
    ["Mapping Status", "Affiliate", "Division Name", "pincode"]
)

# Write output
OUT_COMPARISON = Path("output/Pincode_Mapped_vs_Unmapped_local.csv")
comparison.to_csv(OUT_COMPARISON, index=False)

{
    "total_rows": comparison.shape[0],
    "mapped_rows": (comparison["Mapping Status"] == "Mapped").sum(),
    "unmapped_rows": (comparison["Mapping Status"] == "Unmapped").sum(),
    "output": str(OUT_COMPARISON),
}


{'total_rows': 141854,
 'mapped_rows': np.int64(13487),
 'unmapped_rows': np.int64(128367),
 'output': 'output\\Pincode_Mapped_vs_Unmapped_local.csv'}

In [60]:
# Diagnostics on unmapped rows
unmapped_diag = merged[pd.isna(merged["Final Mapped Territory"])].copy()

unmapped_diag["pincode_issue"] = unmapped_diag["pincode"].isin(
    ["0", "0.0", "INVALID PINCODE"]
)

unmapped_diag["division_missing"] = ~unmapped_diag["Division Name"].isin(
    pin_universe_2["Division Name"].unique()
)

unmapped_diag["affiliate_missing"] = ~unmapped_diag["Affiliate"].isin(
    pin_universe_2["Affiliate"].unique()
)

unmapped_diag[["pincode_issue", "division_missing", "affiliate_missing"]].mean()

pincode_issue        0.000475
division_missing     0.771148
affiliate_missing    0.569196
dtype: float64